# YOLO11n Stage 3 — Targeted Sliced Training + Small/Crowded Fine-Tune + SAHI

This notebook implements the next detector experiment without touching the internal test split.

Pipeline:

**Stage-2 best.pt → targeted train-only slicing → Original + selected slices → 960px small/crowded fine-tune → Standard validation → SAHI validation → comparison**

Current validation reference from Stage 2:

- Small Recall: **0.3611**
- Medium Recall: **0.6745**
- Large Recall: **0.8491**
- Crowded Recall: **0.6568**
- Overall Recall: **0.6949**
- Precision: **0.7885**
- PURE FN: **96**

For direct comparability with the completed audit, this notebook keeps the same operational audit definitions:

- Small: bbox area < **1%** of source image area
- Medium: **1%–<10%**
- Large: **>=10%**
- Crowded image: **>=4 GT objects**

The validation split remains the original 100 images / 354 GT boxes. The test split is not loaded or evaluated.


In [ ]:
# ============================================================
# CELL 1 — INSTALL + MOUNT + ENVIRONMENT
# ============================================================

!pip -q install ultralytics==8.4.117 albumentations==2.0.8 sahi pycocotools pyyaml

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict
import os
import json
import random
import shutil
import time
import math
import yaml

import numpy as np
import pandas as pd
import cv2
import torch
import albumentations as A
import ultralytics

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("=" * 100)
print("ENVIRONMENT")
print("=" * 100)
print("Ultralytics    :", ultralytics.__version__)
print("Albumentations :", A.__version__)
print("PyTorch        :", torch.__version__)
print("CUDA           :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab.")

print("GPU            :", torch.cuda.get_device_name(0))
print("\nSTATUS: ENVIRONMENT_READY")


In [ ]:
# ============================================================
# CELL 2 — PATHS + STRICT SOURCE CHECKS
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/Coral_Reef_AI_Project")

DETECTION_V1 = (
    PROJECT_ROOT
    / "01_Data"
    / "Final"
    / "Detection_v1"
)

TRAIN_IMG_DIR = DETECTION_V1 / "images" / "train"
VAL_IMG_DIR = DETECTION_V1 / "images" / "validation"

TRAIN_COCO = DETECTION_V1 / "annotations" / "instances_train.json"
VAL_COCO = DETECTION_V1 / "annotations" / "instances_validation.json"

SOURCE_BEST = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Enhanced_Underwater_20Epochs"
    / "Detection_Stage2"
    / "weights"
    / "best.pt"
)

EXP_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Stage3_TargetedSliced_SmallCrowded"
)

EXP_ROOT.mkdir(parents=True, exist_ok=True)

# Build locally for fast training.
BUILD_ROOT = Path("/content/Coral_YOLO11_Stage3_TargetedSliced")
SLICE_WORK = Path("/content/Coral_Stage3_Slice_Work")

for p in [
    TRAIN_IMG_DIR,
    VAL_IMG_DIR,
    TRAIN_COCO,
    VAL_COCO,
    SOURCE_BEST,
]:
    if not p.exists():
        raise FileNotFoundError(p)

with open(TRAIN_COCO, "r", encoding="utf-8") as f:
    train_coco = json.load(f)

with open(VAL_COCO, "r", encoding="utf-8") as f:
    val_coco = json.load(f)

print("=" * 100)
print("SOURCE CHECK")
print("=" * 100)
print("Train images:", len(train_coco["images"]))
print("Train boxes :", len(train_coco["annotations"]))
print("Val images  :", len(val_coco["images"]))
print("Val boxes   :", len(val_coco["annotations"]))
print("Stage-2 best:", SOURCE_BEST)

assert len(train_coco["images"]) == 452
assert len(train_coco["annotations"]) == 1540
assert len(val_coco["images"]) == 100
assert len(val_coco["annotations"]) == 354

print("\nSTATUS: SOURCE_DATA_OK")


In [ ]:
# ============================================================
# CELL 3 — RESTORE OLD CUSTOM CLASS NAMES
#
# Required because the Stage-2 checkpoint was serialized while
# these custom Albumentations classes existed in __main__.
# Stage 3 will NOT use these transforms.
# ============================================================

class UnderwaterTurbidity(A.ImageOnlyTransform):

    def __init__(
        self,
        alpha_range=(0.05, 0.16),
        blur_range=(3, 7),
        p=0.15,
    ):
        super().__init__(p=p)
        self.alpha_range = alpha_range
        self.blur_range = blur_range

    def apply(self, img, **params):
        alpha = np.random.uniform(*self.alpha_range)
        veil_value = np.random.randint(125, 180)
        veil = np.full_like(img, veil_value)
        result = cv2.addWeighted(img, 1.0 - alpha, veil, alpha, 0)
        kernel = int(np.random.choice([3, 5, 7]))
        return cv2.GaussianBlur(result, (kernel, kernel), 0)

    def get_transform_init_args_names(self):
        return ("alpha_range", "blur_range")


class UnderwaterBackscatter(A.ImageOnlyTransform):

    def __init__(
        self,
        particles_range=(20, 65),
        radius_range=(1, 3),
        opacity_range=(0.08, 0.20),
        p=0.15,
    ):
        super().__init__(p=p)
        self.particles_range = particles_range
        self.radius_range = radius_range
        self.opacity_range = opacity_range

    def apply(self, img, **params):
        h, w = img.shape[:2]
        overlay = np.zeros_like(img)

        n_particles = np.random.randint(
            self.particles_range[0],
            self.particles_range[1] + 1,
        )

        for _ in range(n_particles):
            x = np.random.randint(0, w)
            y = np.random.randint(0, h)
            radius = np.random.randint(
                self.radius_range[0],
                self.radius_range[1] + 1,
            )
            intensity = np.random.randint(160, 256)
            cv2.circle(
                overlay,
                (x, y),
                radius,
                (intensity, intensity, intensity),
                -1,
            )

        overlay = cv2.GaussianBlur(overlay, (3, 3), 0)
        opacity = np.random.uniform(*self.opacity_range)

        return cv2.addWeighted(
            img,
            1.0,
            overlay,
            opacity,
            0,
        )

    def get_transform_init_args_names(self):
        return (
            "particles_range",
            "radius_range",
            "opacity_range",
        )


print("UnderwaterTurbidity   : READY")
print("UnderwaterBackscatter : READY")
print("\nSTATUS: CHECKPOINT_CLASSES_READY")


In [ ]:
# ============================================================
# CELL 4 — SELECT TRAIN-ONLY SMALL / CROWDED SOURCE IMAGES
# ============================================================

SMALL_AREA_RATIO = 0.01
CROWDED_GT_COUNT = 4

anns_by_image = defaultdict(list)
for ann in train_coco["annotations"]:
    anns_by_image[int(ann["image_id"])].append(ann)

selected_images = []
selection_rows = []

for im in train_coco["images"]:
    image_id = int(im["id"])
    w = float(im["width"])
    h = float(im["height"])
    anns = anns_by_image[image_id]

    rel_areas = []
    for ann in anns:
        _, _, bw, bh = map(float, ann["bbox"])
        rel_areas.append((bw * bh) / max(w * h, 1.0))

    has_small = any(r < SMALL_AREA_RATIO for r in rel_areas)
    is_crowded = len(anns) >= CROWDED_GT_COUNT

    if has_small or is_crowded:
        selected_images.append(im)

    selection_rows.append({
        "image_id": image_id,
        "file_name": Path(im["file_name"]).name,
        "gt_count": len(anns),
        "has_small": has_small,
        "crowded": is_crowded,
        "selected_for_slicing": bool(has_small or is_crowded),
        "min_relative_box_area": min(rel_areas) if rel_areas else np.nan,
    })

selection_df = pd.DataFrame(selection_rows)

selected_ids = {int(im["id"]) for im in selected_images}
selected_anns = [
    ann
    for ann in train_coco["annotations"]
    if int(ann["image_id"]) in selected_ids
]

# Rewrite file_name to basename so SAHI resolves directly inside TRAIN_IMG_DIR.
filtered_images = []
for im in selected_images:
    x = dict(im)
    x["file_name"] = Path(im["file_name"]).name
    filtered_images.append(x)

filtered_coco = {
    "images": filtered_images,
    "annotations": selected_anns,
    "categories": train_coco["categories"],
}

SLICE_WORK.mkdir(parents=True, exist_ok=True)

FILTERED_COCO = SLICE_WORK / "train_small_crowded_sources.json"
with open(FILTERED_COCO, "w", encoding="utf-8") as f:
    json.dump(filtered_coco, f, ensure_ascii=False)

SELECTION_CSV = EXP_ROOT / "stage3_source_selection.csv"
selection_df.to_csv(SELECTION_CSV, index=False)

print("=" * 100)
print("TARGETED TRAIN SOURCE SELECTION")
print("=" * 100)
print("Original train images :", len(train_coco["images"]))
print("Selected for slicing :", len(filtered_images))
print("Selected GT boxes     :", len(selected_anns))
print("Has small images      :", int(selection_df["has_small"].sum()))
print("Crowded images        :", int(selection_df["crowded"].sum()))
print("Union selected        :", int(selection_df["selected_for_slicing"].sum()))

assert set(selected_ids).issubset({int(x["id"]) for x in train_coco["images"]})

print("\nTEST/VALIDATION SOURCES WERE NOT USED FOR SELECTION.")
print("STATUS: TARGET_SOURCES_READY")


In [ ]:
# ============================================================
# CELL 5 — SAHI SLICING ON SELECTED TRAIN SOURCES ONLY
# ============================================================

from sahi.slicing import slice_coco

SLICE_RAW_DIR = SLICE_WORK / "raw_slices"

if SLICE_RAW_DIR.exists():
    shutil.rmtree(SLICE_RAW_DIR)

SLICE_RAW_DIR.mkdir(parents=True, exist_ok=True)

SLICE_SIZE = 640
SLICE_OVERLAP = 0.20

# Require >=40% of a clipped object to remain visible in a tile.
MIN_AREA_RATIO = 0.40

print("=" * 100)
print("RUNNING TARGETED SAHI SLICING")
print("=" * 100)
print("Source images  :", len(filtered_images))
print("Slice size     :", SLICE_SIZE)
print("Overlap        :", SLICE_OVERLAP)
print("Min area ratio :", MIN_AREA_RATIO)

sliced_coco, sliced_coco_path = slice_coco(
    coco_annotation_file_path=str(FILTERED_COCO),
    image_dir=str(TRAIN_IMG_DIR),
    output_coco_annotation_file_name="targeted_sliced_train",
    output_dir=str(SLICE_RAW_DIR),
    ignore_negative_samples=False,
    slice_height=SLICE_SIZE,
    slice_width=SLICE_SIZE,
    overlap_height_ratio=SLICE_OVERLAP,
    overlap_width_ratio=SLICE_OVERLAP,
    min_area_ratio=MIN_AREA_RATIO,
    out_ext=".jpg",
    verbose=False,
    exif_fix=False,
)

print("\nRaw slices :", len(sliced_coco["images"]))
print("Slice boxes :", len(sliced_coco["annotations"]))
print("COCO output :", sliced_coco_path)

print("\nSTATUS: RAW_SLICES_READY")


In [ ]:
# ============================================================
# CELL 6 — CURATE SLICES + BUILD ORIGINAL+SLICED YOLO DATASET
#
# Goals:
# - retain original 452 train images
# - add a controlled number of positive sliced samples
# - retain ~15% negative/background slices
# - keep original 100-image validation untouched
# ============================================================

MAX_POSITIVE_SLICES = 450
NEGATIVE_TO_POSITIVE_RATIO = 0.15

if BUILD_ROOT.exists():
    shutil.rmtree(BUILD_ROOT)

for split in ["train", "val"]:
    (BUILD_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (BUILD_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

slice_anns_by_id = defaultdict(list)
for ann in sliced_coco["annotations"]:
    slice_anns_by_id[int(ann["image_id"])].append(ann)

positive_slice_ids = [
    int(im["id"])
    for im in sliced_coco["images"]
    if len(slice_anns_by_id[int(im["id"])]) > 0
]

negative_slice_ids = [
    int(im["id"])
    for im in sliced_coco["images"]
    if len(slice_anns_by_id[int(im["id"])]) == 0
]

# Keep positives controlled to prevent slices dominating the 452 originals.
rng = random.Random(SEED)
if len(positive_slice_ids) > MAX_POSITIVE_SLICES:
    kept_positive_ids = set(
        rng.sample(positive_slice_ids, MAX_POSITIVE_SLICES)
    )
else:
    kept_positive_ids = set(positive_slice_ids)

n_negative_keep = min(
    len(negative_slice_ids),
    int(round(len(kept_positive_ids) * NEGATIVE_TO_POSITIVE_RATIO)),
)

kept_negative_ids = set(
    rng.sample(negative_slice_ids, n_negative_keep)
    if n_negative_keep > 0
    else []
)

kept_slice_ids = kept_positive_ids | kept_negative_ids

print("=" * 100)
print("SLICE CURATION")
print("=" * 100)
print("Raw positive slices :", len(positive_slice_ids))
print("Raw negative slices :", len(negative_slice_ids))
print("Kept positive       :", len(kept_positive_ids))
print("Kept negatives      :", len(kept_negative_ids))
print("Total kept slices   :", len(kept_slice_ids))


def safe_symlink(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(str(src), str(dst))


def coco_to_yolo_dataset(coco, source_img_dir, out_img_dir, out_label_dir, prefix=""):
    anns_by = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by[int(ann["image_id"])].append(ann)

    image_count = 0
    box_count = 0

    for im in coco["images"]:
        image_id = int(im["id"])
        src_name = Path(im["file_name"]).name
        src = Path(source_img_dir) / src_name

        if not src.exists():
            raise FileNotFoundError(src)

        dst_name = f"{prefix}{src_name}"
        dst = Path(out_img_dir) / dst_name
        safe_symlink(src, dst)

        w = float(im["width"])
        h = float(im["height"])

        lines = []
        for ann in anns_by[image_id]:
            x, y, bw, bh = map(float, ann["bbox"])

            xc = (x + bw / 2.0) / w
            yc = (y + bh / 2.0) / h
            nw = bw / w
            nh = bh / h

            # Single detector class: target_coral -> YOLO class 0.
            lines.append(
                f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
            )
            box_count += 1

        label_path = Path(out_label_dir) / f"{Path(dst_name).stem}.txt"
        label_path.write_text(
            "\n".join(lines) + ("\n" if lines else ""),
            encoding="utf-8",
        )
        image_count += 1

    return image_count, box_count


# ------------------------------------------------------------
# A) ORIGINAL TRAIN
# ------------------------------------------------------------

train_original_count, train_original_boxes = coco_to_yolo_dataset(
    train_coco,
    TRAIN_IMG_DIR,
    BUILD_ROOT / "images" / "train",
    BUILD_ROOT / "labels" / "train",
    prefix="orig__",
)

# ------------------------------------------------------------
# B) ORIGINAL VALIDATION — UNTOUCHED
# ------------------------------------------------------------

val_count, val_boxes = coco_to_yolo_dataset(
    val_coco,
    VAL_IMG_DIR,
    BUILD_ROOT / "images" / "val",
    BUILD_ROOT / "labels" / "val",
    prefix="val__",
)

# ------------------------------------------------------------
# C) TARGETED SLICES
# ------------------------------------------------------------

slice_images_by_id = {
    int(im["id"]): im
    for im in sliced_coco["images"]
}

slice_box_count = 0
slice_rows = []

for slice_id in sorted(kept_slice_ids):
    im = slice_images_by_id[slice_id]
    src_name = Path(im["file_name"]).name
    src = SLICE_RAW_DIR / src_name

    if not src.exists():
        raise FileNotFoundError(src)

    dst_name = f"slice__{src_name}"
    dst = BUILD_ROOT / "images" / "train" / dst_name
    shutil.copy2(src, dst)

    w = float(im["width"])
    h = float(im["height"])

    lines = []
    anns = slice_anns_by_id[slice_id]

    for ann in anns:
        x, y, bw, bh = map(float, ann["bbox"])

        xc = (x + bw / 2.0) / w
        yc = (y + bh / 2.0) / h
        nw = bw / w
        nh = bh / h

        lines.append(
            f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
        )
        slice_box_count += 1

    label_path = BUILD_ROOT / "labels" / "train" / f"{Path(dst_name).stem}.txt"
    label_path.write_text(
        "\n".join(lines) + ("\n" if lines else ""),
        encoding="utf-8",
    )

    slice_rows.append({
        "slice_image": dst_name,
        "slice_id": slice_id,
        "positive": len(anns) > 0,
        "box_count": len(anns),
        "width": int(im["width"]),
        "height": int(im["height"]),
    })

slice_manifest = pd.DataFrame(slice_rows)
SLICE_MANIFEST = EXP_ROOT / "stage3_kept_slices_manifest.csv"
slice_manifest.to_csv(SLICE_MANIFEST, index=False)

# ------------------------------------------------------------
# D) DATA.YAML
# ------------------------------------------------------------

DATA_YAML = BUILD_ROOT / "data.yaml"

data_yaml = {
    "path": str(BUILD_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "target_coral"},
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("\n" + "=" * 100)
print("STAGE-3 DATASET BUILT")
print("=" * 100)
print("Original train images :", train_original_count)
print("Original train boxes  :", train_original_boxes)
print("Added slices          :", len(kept_slice_ids))
print("Added slice boxes     :", slice_box_count)
print("Final train images    :", train_original_count + len(kept_slice_ids))
print("Validation images     :", val_count)
print("Validation boxes      :", val_boxes)
print("data.yaml             :", DATA_YAML)

print("\nSTATUS: STAGE3_DATASET_BUILT")


In [ ]:
# ============================================================
# CELL 7 — STRICT DATASET / LEAKAGE VALIDATION
# ============================================================

train_imgs = sorted((BUILD_ROOT / "images" / "train").glob("*"))
train_labels = sorted((BUILD_ROOT / "labels" / "train").glob("*.txt"))

val_imgs = sorted((BUILD_ROOT / "images" / "val").glob("*"))
val_labels = sorted((BUILD_ROOT / "labels" / "val").glob("*.txt"))

assert len(train_imgs) == len(train_labels)
assert len(val_imgs) == len(val_labels)

assert len(val_imgs) == 100
assert val_boxes == 354
assert train_original_count == 452
assert train_original_boxes == 1540

assert all(p.name.startswith("val__") for p in val_imgs)
assert all(not p.name.startswith("slice__") for p in val_imgs)

train_names = {p.name for p in train_imgs}
val_names = {p.name for p in val_imgs}
assert train_names.isdisjoint(val_names)

def count_yolo_boxes(label_dir):
    n = 0
    invalid = []
    for lp in Path(label_dir).glob("*.txt"):
        text = lp.read_text(encoding="utf-8").strip()
        if not text:
            continue

        for line in text.splitlines():
            parts = line.split()
            if len(parts) != 5:
                invalid.append((lp.name, line))
                continue

            cls, xc, yc, w, h = map(float, parts)

            if int(cls) != 0:
                invalid.append((lp.name, line))

            if not (
                0 <= xc <= 1
                and 0 <= yc <= 1
                and 0 < w <= 1
                and 0 < h <= 1
            ):
                invalid.append((lp.name, line))

            n += 1

    return n, invalid

train_total_boxes, train_invalid = count_yolo_boxes(
    BUILD_ROOT / "labels" / "train"
)
val_total_boxes, val_invalid = count_yolo_boxes(
    BUILD_ROOT / "labels" / "val"
)

assert not train_invalid, train_invalid[:10]
assert not val_invalid, val_invalid[:10]

expected_train_boxes = 1540 + slice_box_count
assert train_total_boxes == expected_train_boxes
assert val_total_boxes == 354

BUILD_SUMMARY = {
    "original_train_images": 452,
    "original_train_boxes": 1540,
    "selected_source_images_for_slicing": len(filtered_images),
    "kept_positive_slices": len(kept_positive_ids),
    "kept_negative_slices": len(kept_negative_ids),
    "added_slices_total": len(kept_slice_ids),
    "added_slice_boxes": slice_box_count,
    "final_train_images": len(train_imgs),
    "final_train_boxes": train_total_boxes,
    "validation_images": len(val_imgs),
    "validation_boxes": val_total_boxes,
    "small_definition_area_ratio_lt": SMALL_AREA_RATIO,
    "crowded_definition_gt_count_gte": CROWDED_GT_COUNT,
    "slice_size": SLICE_SIZE,
    "slice_overlap": SLICE_OVERLAP,
    "slice_min_area_ratio": MIN_AREA_RATIO,
    "test_used": False,
}

BUILD_SUMMARY_PATH = EXP_ROOT / "stage3_dataset_build_summary.json"

with open(BUILD_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(BUILD_SUMMARY, f, indent=2)

print("=" * 100)
print("STRICT DATASET VALIDATION PASSED")
print("=" * 100)

for k, v in BUILD_SUMMARY.items():
    print(f"{k:<42}: {v}")

print("\nNO VALIDATION SLICES WERE CREATED.")
print("NO TEST DATA WAS USED.")
print("\nSTATUS: STAGE3_DATASET_VALIDATED")


In [ ]:
# ============================================================
# CELL 8 — LOAD STAGE-2 BEST + DEFINE MILD STAGE-3 AUGMENTATION
# ============================================================

from ultralytics import YOLO

small_object_augmentations = [
    A.RandomBrightnessContrast(
        brightness_limit=(-0.10, 0.10),
        contrast_limit=(-0.12, 0.12),
        p=0.20,
    ),
    A.CLAHE(
        clip_limit=(1.0, 1.8),
        tile_grid_size=(8, 8),
        p=0.04,
    ),
]

print("=" * 100)
print("LOADING STAGE-2 BEST")
print("=" * 100)

model = YOLO(str(SOURCE_BEST))

print("MODEL LOADED")

print("\nStage-3 custom image degradation:")
print("Motion/Gaussian blur : OFF")
print("Noise                : OFF")
print("JPEG compression     : OFF")
print("Downscale            : OFF")
print("Heavy turbidity      : OFF")
print("Heavy backscatter    : OFF")
print("Brightness/contrast  : MILD")
print("CLAHE                : VERY MILD")

print("\nSTATUS: STAGE3_MODEL_READY")


In [ ]:
# ============================================================
# CELL 9 — SMALL/CROWDED FOCUSED FINE-TUNE
# ============================================================

RUN_NAME = "Detection_Stage3"

print("=" * 110)
print("YOLO11n STAGE 3 — TARGETED SLICED SMALL/CROWDED FINE-TUNE")
print("=" * 110)
print("Starting weights : Stage-2 enhanced best.pt")
print("Training data    : Original 452 + targeted slices")
print("Validation       : Original 100 images only")
print("Epochs           : 12")
print("imgsz            : 960")
print("Batch            : 8")
print("LR               : 0.00015")
print("Multi-scale      : OFF")
print("Mosaic           : 0.10")
print("MixUp/CutMix     : OFF")
print("Test             : NOT USED")
print("=" * 110)

start_time = time.time()

train_result = model.train(
    data=str(DATA_YAML),

    epochs=12,
    imgsz=960,
    batch=8,
    device=0,
    workers=2,

    optimizer="AdamW",
    lr0=0.00015,
    lrf=0.10,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=0.5,

    multi_scale=0.0,

    degrees=5.0,
    translate=0.05,
    scale=0.10,
    shear=0.0,
    perspective=0.0,
    fliplr=0.50,
    flipud=0.05,

    mosaic=0.10,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    close_mosaic=4,

    hsv_h=0.004,
    hsv_s=0.10,
    hsv_v=0.08,

    augmentations=small_object_augmentations,

    # Keep standard detector loss weights:
    # audit showed missed small objects, not a small-object localization failure.
    box=7.5,
    cls=0.5,
    dfl=1.5,

    amp=True,
    seed=SEED,
    deterministic=True,
    patience=100,
    cache=False,

    save=True,
    save_period=3,
    plots=True,

    project=str(EXP_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)

elapsed_min = (time.time() - start_time) / 60.0

RUN_DIR = Path(train_result.save_dir)
BEST_STAGE3 = RUN_DIR / "weights" / "best.pt"
LAST_STAGE3 = RUN_DIR / "weights" / "last.pt"

assert BEST_STAGE3.exists()

print("\n" + "=" * 100)
print("STAGE-3 TRAINING COMPLETE")
print("=" * 100)
print(f"Time     : {elapsed_min:.2f} min")
print("Run dir  :", RUN_DIR)
print("best.pt  :", BEST_STAGE3)
print("last.pt  :", LAST_STAGE3)
print("\nSTATUS: STAGE3_TRAINING_COMPLETE")


In [ ]:
# ============================================================
# CELL 10 — STANDARD VALIDATION ON ORIGINAL 100 IMAGES
# ============================================================

stage3_model = YOLO(str(BEST_STAGE3))

standard_ultra_metrics = stage3_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=960,
    batch=8,
    device=0,
    conf=0.001,
    iou=0.70,
    max_det=300,
    plots=True,
    verbose=True,
)

print("\n" + "=" * 100)
print("STAGE-3 STANDARD ULTRALYTICS VALIDATION")
print("=" * 100)
print("Precision :", float(standard_ultra_metrics.box.mp))
print("Recall    :", float(standard_ultra_metrics.box.mr))
print("mAP50     :", float(standard_ultra_metrics.box.map50))
print("mAP50:95  :", float(standard_ultra_metrics.box.map))
print("\nTEST WAS NOT EVALUATED.")
print("\nSTATUS: STANDARD_VALIDATION_COMPLETE")


In [ ]:
# ============================================================
# CELL 11 — COMMON CUSTOM AUDIT FUNCTIONS
# Used for fair Standard vs SAHI small/crowded comparison.
# ============================================================

with open(VAL_COCO, "r", encoding="utf-8") as f:
    audit_val_coco = json.load(f)

val_image_id_by_basename = {
    Path(im["file_name"]).name: int(im["id"])
    for im in audit_val_coco["images"]
}

val_image_meta = {
    int(im["id"]): im
    for im in audit_val_coco["images"]
}

val_gt_by_id = defaultdict(list)

for ann in audit_val_coco["annotations"]:
    image_id = int(ann["image_id"])
    x, y, w, h = map(float, ann["bbox"])

    val_gt_by_id[image_id].append({
        "bbox": np.array([x, y, x + w, y + h], dtype=np.float32),
        "area": float(w * h),
    })


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)

    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)

    inter = wh[..., 0] * wh[..., 1]

    aa = (
        np.clip(a[:, 2] - a[:, 0], 0, None)
        * np.clip(a[:, 3] - a[:, 1], 0, None)
    )

    bb = (
        np.clip(b[:, 2] - b[:, 0], 0, None)
        * np.clip(b[:, 3] - b[:, 1], 0, None)
    )

    union = aa[:, None] + bb[None, :] - inter

    return inter / np.clip(union, 1e-9, None)


def greedy_match(pred_boxes, gt_boxes, threshold=0.50):
    if len(pred_boxes) == 0:
        return [], [], list(range(len(gt_boxes)))

    if len(gt_boxes) == 0:
        return [], list(range(len(pred_boxes))), []

    ious = iou_matrix(pred_boxes, gt_boxes)

    candidates = []
    for p in range(len(pred_boxes)):
        for g in range(len(gt_boxes)):
            if ious[p, g] >= threshold:
                candidates.append((float(ious[p, g]), p, g))

    candidates.sort(reverse=True)

    used_p, used_g, matches = set(), set(), []

    for iou, p, g in candidates:
        if p in used_p or g in used_g:
            continue
        used_p.add(p)
        used_g.add(g)
        matches.append((p, g, iou))

    up = [p for p in range(len(pred_boxes)) if p not in used_p]
    ug = [g for g in range(len(gt_boxes)) if g not in used_g]

    return matches, up, ug


def size_bucket(gt_box, image_w, image_h):
    x1, y1, x2, y2 = map(float, gt_box)
    ratio = (
        max(0.0, x2 - x1) * max(0.0, y2 - y1)
        / max(float(image_w * image_h), 1.0)
    )

    if ratio < 0.01:
        return "small"
    if ratio < 0.10:
        return "medium"
    return "large"


def threshold_sweep(pred_dict):
    rows = []

    for conf in np.round(np.arange(0.05, 0.81, 0.025), 3):
        TP = FP = FN = 0

        for image_name, pred in pred_dict.items():
            image_id = val_image_id_by_basename[image_name]

            gt_boxes = np.array(
                [x["bbox"] for x in val_gt_by_id[image_id]],
                dtype=np.float32,
            ).reshape(-1, 4)

            keep = pred["conf"] >= conf
            pb = pred["boxes"][keep]

            matches, up, ug = greedy_match(pb, gt_boxes, 0.50)

            TP += len(matches)
            FP += len(up)
            FN += len(ug)

        p = TP / (TP + FP) if (TP + FP) else 0.0
        r = TP / (TP + FN) if (TP + FN) else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0

        rows.append({
            "confidence": float(conf),
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "precision": p,
            "recall": r,
            "f1": f1,
        })

    df = pd.DataFrame(rows)
    best = df.loc[df["f1"].idxmax()]
    return df, best


def detailed_metrics(pred_dict, conf):
    gt_rows = []
    total_tp = total_fp = total_fn = 0

    crowded_totals = {
        False: {"TP": 0, "FP": 0, "FN": 0, "GT": 0, "images": 0},
        True: {"TP": 0, "FP": 0, "FN": 0, "GT": 0, "images": 0},
    }

    for image_name, pred in pred_dict.items():
        image_id = val_image_id_by_basename[image_name]
        meta = val_image_meta[image_id]

        gt_items = val_gt_by_id[image_id]
        gt_boxes = np.array(
            [x["bbox"] for x in gt_items],
            dtype=np.float32,
        ).reshape(-1, 4)

        keep = pred["conf"] >= conf
        pb = pred["boxes"][keep]

        matches, up, ug = greedy_match(pb, gt_boxes, 0.50)
        matched_gt = {g: (p, iou) for p, g, iou in matches}

        for g, gt in enumerate(gt_items):
            bucket = size_bucket(
                gt["bbox"],
                int(meta["width"]),
                int(meta["height"]),
            )

            gt_rows.append({
                "image": image_name,
                "size": bucket,
                "detected": g in matched_gt,
                "iou": float(matched_gt[g][1]) if g in matched_gt else 0.0,
            })

        tp = len(matches)
        fp = len(up)
        fn = len(ug)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        crowded = len(gt_items) >= 4
        crowded_totals[crowded]["TP"] += tp
        crowded_totals[crowded]["FP"] += fp
        crowded_totals[crowded]["FN"] += fn
        crowded_totals[crowded]["GT"] += len(gt_items)
        crowded_totals[crowded]["images"] += 1

    gt_df = pd.DataFrame(gt_rows)

    size_rows = []
    for bucket in ["small", "medium", "large"]:
        g = gt_df[gt_df["size"] == bucket]
        recall = float(g["detected"].mean()) if len(g) else np.nan
        size_rows.append({
            "size": bucket,
            "GT": len(g),
            "TP": int(g["detected"].sum()),
            "FN": int((~g["detected"]).sum()),
            "recall": recall,
        })

    crowded_rows = []
    for crowded in [False, True]:
        d = crowded_totals[crowded]
        p = d["TP"] / (d["TP"] + d["FP"]) if (d["TP"] + d["FP"]) else 0.0
        r = d["TP"] / (d["TP"] + d["FN"]) if (d["TP"] + d["FN"]) else 0.0

        crowded_rows.append({
            "crowded": crowded,
            "images": d["images"],
            "GT": d["GT"],
            "TP": d["TP"],
            "FP": d["FP"],
            "FN": d["FN"],
            "precision": p,
            "recall": r,
        })

    p_all = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    r_all = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    f1_all = 2 * p_all * r_all / (p_all + r_all) if (p_all + r_all) else 0.0

    overall = {
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "precision": p_all,
        "recall": r_all,
        "f1": f1_all,
    }

    return overall, pd.DataFrame(size_rows), pd.DataFrame(crowded_rows), gt_df


print("STATUS: AUDIT_FUNCTIONS_READY")


In [ ]:
# ============================================================
# CELL 12 — STAGE-3 STANDARD INFERENCE CUSTOM AUDIT
# ============================================================

original_val_paths = sorted([
    p for p in VAL_IMG_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
])

assert len(original_val_paths) == 100

standard_results = stage3_model.predict(
    source=[str(p) for p in original_val_paths],
    imgsz=960,
    conf=0.05,
    iou=0.70,
    max_det=300,
    device=0,
    verbose=False,
)

standard_predictions = {}

for image_path, r in zip(original_val_paths, standard_results):
    image_name = image_path.name

    if r.boxes is None or len(r.boxes) == 0:
        boxes = np.empty((0, 4), dtype=np.float32)
        conf = np.empty((0,), dtype=np.float32)
    else:
        boxes = r.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
        conf = r.boxes.conf.detach().cpu().numpy().astype(np.float32)

    standard_predictions[image_name] = {
        "boxes": boxes,
        "conf": conf,
    }

standard_sweep, standard_best = threshold_sweep(standard_predictions)

STANDARD_BEST_CONF = float(standard_best["confidence"])

(
    standard_overall,
    standard_size,
    standard_crowded,
    standard_gt_df,
) = detailed_metrics(
    standard_predictions,
    STANDARD_BEST_CONF,
)

print("=" * 100)
print("STAGE-3 STANDARD CUSTOM AUDIT")
print("=" * 100)
print("Best conf:", STANDARD_BEST_CONF)
print("Overall :", standard_overall)
print("\nBy size:")
print(standard_size.to_string(index=False))
print("\nCrowded:")
print(standard_crowded.to_string(index=False))

print("\nSTATUS: STANDARD_CUSTOM_AUDIT_COMPLETE")


In [ ]:
# ============================================================
# CELL 13 — SAHI INFERENCE ON ORIGINAL 100 VALIDATION IMAGES
#
# 640x640 slices, 20% overlap, plus standard full-image pass.
# ============================================================

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.postprocess.backends import set_postprocess_backend

# GPU-accelerated postprocessing when available.
try:
    set_postprocess_backend("torchvision")
    print("SAHI postprocess backend: torchvision")
except Exception as e:
    print("Could not force torchvision backend:", e)
    set_postprocess_backend("auto")
    print("SAHI postprocess backend: auto")

SAHI_SLICE = 640
SAHI_OVERLAP = 0.20
SAHI_MIN_CONF = 0.05

# Custom checkpoint classes were already restored in CELL 3.
sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(BEST_STAGE3),
    confidence_threshold=SAHI_MIN_CONF,
    device="cuda:0",
    image_size=960,
)

sahi_predictions = {}
sahi_latency_rows = []

print("=" * 100)
print("RUNNING SAHI VALIDATION")
print("=" * 100)
print("Images       :", len(original_val_paths))
print("Slice        :", SAHI_SLICE)
print("Overlap      :", SAHI_OVERLAP)
print("Model input  : 960")
print("Standard pass: ON")

for idx, image_path in enumerate(original_val_paths, start=1):
    t0 = time.time()

    result = get_sliced_prediction(
        str(image_path),
        sahi_model,
        slice_height=SAHI_SLICE,
        slice_width=SAHI_SLICE,
        overlap_height_ratio=SAHI_OVERLAP,
        overlap_width_ratio=SAHI_OVERLAP,
        perform_standard_pred=True,
        postprocess_type="GREEDYNMM",
        postprocess_match_metric="IOU",
        postprocess_match_threshold=0.50,
        batch_size=8,
        progress_bar=False,
        verbose=0,
    )

    dt = time.time() - t0

    boxes = []
    confs = []

    for pred in result.object_prediction_list:
        boxes.append(pred.bbox.to_xyxy())
        confs.append(float(pred.score.value))

    boxes = (
        np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
        if boxes
        else np.empty((0, 4), dtype=np.float32)
    )

    confs = (
        np.asarray(confs, dtype=np.float32)
        if confs
        else np.empty((0,), dtype=np.float32)
    )

    sahi_predictions[image_path.name] = {
        "boxes": boxes,
        "conf": confs,
    }

    sahi_latency_rows.append({
        "image": image_path.name,
        "seconds": dt,
        "predictions": len(boxes),
    })

    if idx % 10 == 0 or idx == len(original_val_paths):
        print(f"{idx:3d}/{len(original_val_paths)} images complete")

sahi_latency_df = pd.DataFrame(sahi_latency_rows)

print("\nMean SAHI seconds/image :", sahi_latency_df["seconds"].mean())
print("Median seconds/image    :", sahi_latency_df["seconds"].median())

print("\nSTATUS: SAHI_PREDICTIONS_READY")


In [ ]:
# ============================================================
# CELL 14 — SAHI CUSTOM AUDIT + STAGE2/STAGE3/SAHI COMPARISON
# ============================================================

sahi_sweep, sahi_best = threshold_sweep(sahi_predictions)
SAHI_BEST_CONF = float(sahi_best["confidence"])

(
    sahi_overall,
    sahi_size,
    sahi_crowded,
    sahi_gt_df,
) = detailed_metrics(
    sahi_predictions,
    SAHI_BEST_CONF,
)

print("=" * 100)
print("STAGE-3 + SAHI CUSTOM AUDIT")
print("=" * 100)
print("Best conf:", SAHI_BEST_CONF)
print("Overall :", sahi_overall)
print("\nBy size:")
print(sahi_size.to_string(index=False))
print("\nCrowded:")
print(sahi_crowded.to_string(index=False))

# Stage-2 reference from completed validation audit.
stage2_ref = {
    "method": "Stage2 Standard",
    "precision": 0.7884615385,
    "overall_recall": 0.6949152542,
    "small_recall": 0.3611111111,
    "medium_recall": 0.6745283019,
    "large_recall": 0.8490566038,
    "crowded_recall": 0.6567796610,
    "FN": 96,
    "FP": 54,
    "best_conf": 0.325,
}

def row_from_results(method, overall, size_df, crowded_df, best_conf):
    small = float(size_df.loc[size_df["size"] == "small", "recall"].iloc[0])
    medium = float(size_df.loc[size_df["size"] == "medium", "recall"].iloc[0])
    large = float(size_df.loc[size_df["size"] == "large", "recall"].iloc[0])
    crowded = float(
        crowded_df.loc[crowded_df["crowded"] == True, "recall"].iloc[0]
    )

    return {
        "method": method,
        "precision": overall["precision"],
        "overall_recall": overall["recall"],
        "small_recall": small,
        "medium_recall": medium,
        "large_recall": large,
        "crowded_recall": crowded,
        "FN": overall["FN"],
        "FP": overall["FP"],
        "best_conf": best_conf,
    }

comparison_rows = [
    stage2_ref,
    row_from_results(
        "Stage3 Standard",
        standard_overall,
        standard_size,
        standard_crowded,
        STANDARD_BEST_CONF,
    ),
    row_from_results(
        "Stage3 + SAHI640",
        sahi_overall,
        sahi_size,
        sahi_crowded,
        SAHI_BEST_CONF,
    ),
]

comparison_df = pd.DataFrame(comparison_rows)

COMPARISON_CSV = EXP_ROOT / "stage2_stage3_sahi_comparison.csv"
comparison_df.to_csv(COMPARISON_CSV, index=False)

STANDARD_SWEEP_CSV = EXP_ROOT / "stage3_standard_threshold_sweep.csv"
SAHI_SWEEP_CSV = EXP_ROOT / "stage3_sahi_threshold_sweep.csv"
SAHI_LATENCY_CSV = EXP_ROOT / "stage3_sahi_latency.csv"

standard_sweep.to_csv(STANDARD_SWEEP_CSV, index=False)
sahi_sweep.to_csv(SAHI_SWEEP_CSV, index=False)
sahi_latency_df.to_csv(SAHI_LATENCY_CSV, index=False)

print("\n" + "=" * 120)
print("FINAL COMPARISON")
print("=" * 120)

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "overall_recall": "{:.4f}".format,
            "small_recall": "{:.4f}".format,
            "medium_recall": "{:.4f}".format,
            "large_recall": "{:.4f}".format,
            "crowded_recall": "{:.4f}".format,
            "best_conf": "{:.3f}".format,
        },
    )
)

print("\nSaved:", COMPARISON_CSV)
print("Mean SAHI sec/image:", float(sahi_latency_df["seconds"].mean()))

print("\nTEST SET WAS NOT USED.")
print("\nSTATUS: STAGE3_AND_SAHI_EXPERIMENT_COMPLETE")


## Decision rule after the run

The primary success criteria are not only overall mAP. Check the final comparison for:

- **Small Recall**: target > 0.50
- **Crowded Recall**: target > 0.70
- **Overall Recall**: target >= 0.72
- **FN**: target < 80
- **Precision**: preferably >= 0.75

If Stage 3 Standard improves small/crowded recall, the training intervention worked.

If Stage 3 + SAHI improves further but latency is too high, keep two operating modes:

**Real-time mode = Stage 3 Standard**  
**High-accuracy mode = Stage 3 + SAHI**

Do not evaluate the internal test split until the detector configuration is frozen.
